## LoanShield — Part 1: EDA & Fraud Landscape
Ensemble Methods: Decision Tree vs Random Forest vs XGBoost

Dataset : HMEQ (Home Equity Loan Default) — 5,960 rows

Domain  : FinTech / Lending Risk

This notebook covers data loading, exploratory analysis, cleaning, preprocessing, and the train/test split. It ends by saving the processed artifacts to disk so **02_modeling_ensemble_verdict.ipynb** can pick up from here without re-running this pipeline.


##CELL 1 - Import Requried  Library

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection      import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing        import LabelEncoder, StandardScaler
from sklearn.tree                 import DecisionTreeClassifier
from sklearn.ensemble             import RandomForestClassifier
from xgboost                      import XGBClassifier
from sklearn.metrics              import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)


In [ ]:
#  Dark-theme plot settings (consistent with portfolio)
plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a2e",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#e0e0e0",
    "xtick.color":      "#aaa",
    "ytick.color":      "#aaa",
    "text.color":       "#e0e0e0",
    "grid.color":       "#2a2a4a",
    "figure.dpi":       150,
    "legend.facecolor": "#1a1a2e",
    "legend.edgecolor": "#444",
})

In [ ]:
#  Colour palette
COLOR_FRAUD   = "#ef5350"   # red   — fraudulent / high-risk
COLOR_LEGIT   = "#66bb6a"   # green — legitimate / low-risk
COLOR_DT      = "#78909c"   # gray  — baseline Decision Tree
COLOR_RF      = "#42a5f5"   # blue  — Random Forest
COLOR_XGB     = "#ffca28"   # amber — XGBoost
COLOR_NEUTRAL = "#7e57c2"   # purple — neutral highlights


# CELL 2 — Load Dataset
#
   Go to: https://www.kaggle.com/datasets/ajay1735/hmeq-data and downlode the dataset.

   Lode dataset into the colab notebook.



In [ ]:
print("  LOADING DATA\n")
import os
def find_file(filename):
    if os.path.exists(filename):
        return filename
    # Search in subdirectories
    for root, dirs, files in os.walk('.'):
        if filename in files:
            return os.path.join(root, filename)
    # Search in parent directories up to 3 levels
    path = '..'
    for _ in range(3):
        target = os.path.abspath(os.path.join(path, filename))
        if os.path.exists(target):
            return target
        path = os.path.join(path, '..')
    return filename
raw_df = pd.read_csv(find_file("hmeq.csv"))
print(f"2.1  Shape   : {raw_df.shape}")

In [ ]:
print(f"2.2 Columns : {list(raw_df.columns)}")

In [ ]:
print(f"2.3  First 3 rows:\n")
print(raw_df.head(3).to_string())

In [ ]:
print(f"2.4 Target distribution (BAD):")
print(raw_df["BAD"].value_counts())

In [ ]:
print(f"2.5 Fraud rate: {raw_df['BAD'].mean()*100:.2f}%")

In [ ]:
print(f"2.6 Statistical analysic of dataset")
print(raw_df.describe().to_string())

In [ ]:
print(f"2.7 Basic info about dataset ")
print(raw_df.info())


# CELL 3 —  EDA & Missing Value Audit

 WHY: Before building any model, a lender needs to understand the risk
      landscape. Three questions:

        1. How rare is fraud?
        2. What do fraudsters look like vs legitimate borrowers?
        3. Which features are dirty and need cleaning?

In [ ]:
df = raw_df.copy()

# 3.1  Basic audit
print("\n[3.1]  Shape:", df.shape)
print("\n[3.2]  Missing values per column:")
missing     = df.isnull().sum()
print(missing[missing > 0].to_string())

In [ ]:
# 3.3  Class distribution
fraud_count = df["BAD"].sum()
legit_count = len(df) - fraud_count
fraud_rate  = fraud_count / len(df) * 100
print(f"[3.3]  Class distribution:\n")
print(f"Legitimate (BAD=0)    : {legit_count:,}  ({100-fraud_rate:.1f}%)")
print(f"Default/Fraud (BAD=1) : {fraud_count:,}  ({fraud_rate:.1f}%)")

In [ ]:
# 3.4  Numeric feature statistics split by class

print("[3.4]  Key feature means — fraud vs legitimate:\n")
num_feats = df.select_dtypes(include=np.number).columns
summary   = df.groupby("BAD")[num_feats].mean().round(2)
summary.index = ["Legitimate", "Fraud/Default"]
print(summary.to_string())

# CELL 4 - Visulation act 1

In [ ]:
# 4.1 Class Distribution

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Legitimate\n(BAD=0)", "Default / Fraud\n(BAD=1)"],
              [legit_count, fraud_count],
              color=[COLOR_LEGIT, COLOR_FRAUD], width=0.5, edgecolor="#333")
ax.set_title("Class Distribution — How Rare Is Fraud?", color="#e0e0e0")
ax.set_ylabel("Number of Applications")
for bar, count in zip(bars, [legit_count, fraud_count]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 40,
            f"{count:,}\n({count/len(df)*100:.1f}%)",
            ha="center", va="bottom", color="#e0e0e0", fontsize=11, fontweight="bold")
ax.set_ylim(0, legit_count * 1.2)
ax.text(0.98, 0.95,
        f"Imbalance ratio\n{legit_count/fraud_count:.1f} : 1",
        transform=ax.transAxes, ha="right", va="top",
        color="#ffca28", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#2a2a4a", edgecolor="#ffca28"))
plt.tight_layout()
plt.show()
print("✓[4.1] Panel 1 — Class Distribution ")

In [ ]:
# 4.2 Debt-to-Income Distribution

fig, ax = plt.subplots(figsize=(6, 4))
legit_di = df[df["BAD"] == 0]["DEBTINC"].dropna()
fraud_di = df[df["BAD"] == 1]["DEBTINC"].dropna()
ax.hist(legit_di, bins=50, alpha=0.65, color=COLOR_LEGIT,
        label=f"Legitimate (n={len(legit_di):,})", density=True)
ax.hist(fraud_di, bins=50, alpha=0.65, color=COLOR_FRAUD,
        label=f"Default/Fraud (n={len(fraud_di):,})", density=True)
ax.axvline(legit_di.mean(), color=COLOR_LEGIT, lw=2, ls="--",
           label=f"Legit mean: {legit_di.mean():.1f}%")
ax.axvline(fraud_di.mean(), color=COLOR_FRAUD, lw=2, ls="--",
           label=f"Fraud mean: {fraud_di.mean():.1f}%")
ax.set_title("Debt-to-Income Ratio — Key Risk Signal", color="#e0e0e0")
ax.set_xlabel("Debt-to-Income (%)")
ax.set_ylabel("Density")
ax.legend(fontsize=8, framealpha=0.4)
plt.tight_layout()
plt.show()
print("✓[4.2] Panel 2 — Debt-to-Income Distribution ")

In [ ]:
# 4.3 Miing Value Heatmap
fig, ax = plt.subplots(figsize=(6, 4))
missing_by_class = df.groupby("BAD").apply(lambda x: x.isnull().mean() * 100)
missing_by_class.index = ["Legitimate", "Fraud/Default"]
missing_by_class = missing_by_class.drop(columns=["BAD"], errors="ignore")
missing_by_class = missing_by_class.loc[:, missing_by_class.max() > 0]
im2 = ax.imshow(missing_by_class.values, aspect="auto", cmap="YlOrRd",
                vmin=0, vmax=missing_by_class.values.max())
ax.set_xticks(range(len(missing_by_class.columns)))
ax.set_xticklabels(missing_by_class.columns, rotation=35, ha="right", fontsize=9)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Legitimate", "Fraud/Default"], fontsize=10)
ax.set_title("Missing Data % by Class — Imputation Needed", color="#e0e0e0")
for i in range(2):
    for j in range(len(missing_by_class.columns)):
        ax.text(j, i, f"{missing_by_class.values[i, j]:.1f}%",
                ha="center", va="center", color="#111", fontsize=8, fontweight="bold")
plt.colorbar(im2, ax=ax, label="% Missing")
plt.tight_layout()
plt.show()
print("✓ 4.3 Panel  — Miing Value Heatmapss ")

# CELL 5 — Data Cleaning & Preprocessing


In [ ]:
print("  DATA CLEANING & PREPROCESSING\n")

df_clean = df.copy()

# 5.1  Categorical columns — fill with mode
for col in ["REASON", "JOB"]:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"[5.1]  {col:7s} — filled {df[col].isnull().sum()} NaNs ")

In [ ]:
# 5.2  Numeric columns — class-conditional median imputation

num_cols_with_na = [c for c in df_clean.select_dtypes(include=np.number).columns
                     if df_clean[c].isnull().sum() > 0]

for col in num_cols_with_na:
    for cls in [0, 1]:
        median_val = df_clean.loc[df_clean["BAD"] == cls, col].median()
        mask       = (df_clean["BAD"] == cls) & (df_clean[col].isnull())
        df_clean.loc[mask, col] = median_val
    n_filled = df[col].isnull().sum()
    print(f"[5.2]  {col:8s} — class-conditional median imputation ({n_filled} NaNs filled)")

In [ ]:
# 5.3  Verify zero NaNs

assert df_clean.isnull().sum().sum() == 0, "FATAL: residual NaNs after cleaning"
print(f"\n[5.3]  ✓ Zero NaNs confirmed — shape: {df_clean.shape}")

In [ ]:
# 5.4  Encode categorical features

print(" [5.4] Categorical Label Encoding ")
le_reason = LabelEncoder()
le_job    = LabelEncoder()
df_clean["REASON_ENC"] = le_reason.fit_transform(df_clean["REASON"])
df_clean["JOB_ENC"]    = le_job.fit_transform(df_clean["JOB"])

print(f"\n REASON categories : {list(le_reason.classes_)}")
print(f" JOB    categories : {list(le_job.classes_)}")

In [ ]:
# 5.5  Feature matrix & target vector
FEATURE_COLS = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ",
                "CLAGE", "NINQ", "CLNO", "DEBTINC", "REASON_ENC", "JOB_ENC"]

X = df_clean[FEATURE_COLS]
y = df_clean["BAD"]

print(f"\n[5.5]  Feature matrix shape : {X.shape}")
print(f"       Target vector shape  : {y.shape}\n")
print(f"       Feature names        : {FEATURE_COLS}")

# CELL 6 — Train / Test Split (Stratified)

In [ ]:
print("  TRAIN / TEST SPLIT")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y          # preserves fraud rate in both splits
)

print(f"\n  Train size : {X_train.shape[0]:,} rows  (fraud rate: {y_train.mean()*100:.1f}%)")
print(f"  Test  size : {X_test.shape[0]:,}  rows  (fraud rate: {y_test.mean()*100:.1f}%)")
print(f"\n  ✓ Stratification confirmed — fraud rate preserved in both splits")

In [ ]:
# Save processed artifacts for the modeling notebook
import pickle

artifacts = {
    "df_clean": df_clean,
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "y": y,
    "FEATURE_COLS": FEATURE_COLS,
}

with open("loanshield_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("✓ Saved processed artifacts to loanshield_artifacts.pkl")
print(f"  Keys: {list(artifacts.keys())}")
